In [15]:
import numpy as np
import pandas as pd

np.random.seed(42)
# Load hcp_master from the leakage-fixed aggregation
n = len(df_out)  # df_out comes from cell 1

crm = pd.DataFrame({
    'Prscrbr_NPI'        : df_out['Prscrbr_NPI'],
    'emails_sent'        : np.random.randint(0, 20, n),
    'emails_opened'      : np.random.randint(0, 10, n),
    'rep_visits'         : np.random.randint(0, 10, n),
    'rep_visit_accepted' : np.random.randint(0, 5, n),
    'webinars_invited'   : np.random.randint(0, 5, n),
    'webinars_attended'  : np.random.randint(0, 3, n),
})
crm['emails_opened']      = crm[['emails_sent','emails_opened']].min(axis=1)
crm['rep_visit_accepted'] = crm[['rep_visits','rep_visit_accepted']].min(axis=1)
crm['webinars_attended']  = crm[['webinars_invited','webinars_attended']].min(axis=1)
crm['digital_score'] = (
    (crm['emails_opened'] / (crm['emails_sent'] + 1)) * 50 +
    (crm['webinars_attended'] / (crm['webinars_invited'] + 1)) * 50
).round(2)
crm.to_csv('crm_layer.csv', index=False)
print(f"CRM layer saved: {crm.shape}")

CRM layer saved: (625089, 8)


In [36]:
import pandas as pd

df = pd.read_csv('hcp_final.csv')

# Recreate and save priority_score permanently
segment_weight = {
    'Champions': 1.0, 'GLP1_Rising': 0.9, 'Digital_First': 0.6,
    'Low_Priority': 0.2, 'Dormant': 0.1
}
df['segment_weight'] = df['segment_name'].map(segment_weight).fillna(0.1)
df['priority_score'] = (
    df['propensity_score'] * 0.5 +
    df['segment_weight'] * 0.3 +
    df['is_kol'] * 0.2
)

df.to_csv('hcp_final.csv', index=False)
print(f"Saved. Columns: {df.columns.tolist()}")

Saved. Columns: ['Prscrbr_NPI', 'total_claims', 'total_patients', 'total_drug_cost', 'unique_drugs', 'specialty', 'city', 'state', 'glp1_claims', 'glp1_patients', 'prescribes_glp1', 'emails_sent', 'emails_opened', 'rep_visits', 'rep_visit_accepted', 'webinars_invited', 'webinars_attended', 'digital_score', 'segment', 'segment_name', 'propensity_score', 'pagerank', 'is_kol', 'next_best_action', 'segment_weight', 'priority_score']


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix

# 1. Load Raw Data
df = pd.read_csv('data2024.csv', low_memory=False)

# 2. Identify GLP-1 vs Non-GLP-1 records
glp1_mask = df['Gnrc_Name'].str.contains('semaglutide|tirzepatide|dulaglutide|liraglutide|exenatide', case=False, na=False)

# 3. Create Target from GLP-1 Data
df_glp1 = df[glp1_mask]
glp1_prescribers = df_glp1.groupby('Prscrbr_NPI').agg(
    glp1_claims=('Tot_Clms', 'sum'),
    glp1_patients=('Tot_Benes', 'sum')
).reset_index()
glp1_prescribers['prescribes_glp1'] = 1

# 4. FIX SECONDARY LEAKAGE: Aggregate baseline volume features ONLY from Non-GLP-1 Data
df_non_glp1 = df[~glp1_mask]
hcp_features = df_non_glp1.groupby('Prscrbr_NPI').agg(
    total_claims=('Tot_Clms', 'sum'),
    total_patients=('Tot_Benes', 'sum'),
    total_drug_cost=('Tot_Drug_Cst', 'sum'),
    unique_drugs=('Gnrc_Name', 'nunique'),
    specialty=('Prscrbr_Type', 'first'),
    city=('Prscrbr_City', 'first'),
    state=('Prscrbr_State_Abrvtn', 'first')
).reset_index()

# Free up memory
del df, df_glp1, df_non_glp1

# 5. Merge Baseline Features with Target
hcp_master = hcp_features.merge(glp1_prescribers, on='Prscrbr_NPI', how='left')
hcp_master['prescribes_glp1'] = hcp_master['prescribes_glp1'].fillna(0).astype(int)
hcp_master['glp1_claims'] = hcp_master['glp1_claims'].fillna(0)
hcp_master['glp1_patients'] = hcp_master['glp1_patients'].fillna(0)
hcp_master = hcp_master.dropna(subset=['specialty'])

# 6. Re-attach CRM Data
crm = pd.read_csv('crm_layer.csv')
df_model_base = hcp_master.merge(crm, on='Prscrbr_NPI', how='left')

# 7. FIX PRIMARY LEAKAGE: Remove segment_name. One-hot encode ONLY specialty.
df_model = pd.get_dummies(df_model_base, columns=['specialty'], drop_first=True)

# 8. Define strict feature list (Target and Target-proxies completely removed)
drop_cols = ['Prscrbr_NPI', 'city', 'state', 'prescribes_glp1', 'glp1_claims', 'glp1_patients', 'segment', 'segment_name', 'segment_weight']
feature_cols = [c for c in df_model.columns if c not in drop_cols]

X = df_model[feature_cols].fillna(0)
y = df_model['prescribes_glp1']

# 9. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 10. Retrain XGBoost
scale = y_train.value_counts()[0] / y_train.value_counts()[1]
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale,
    use_label_encoder=False,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# 11. Generate Honest Metrics
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("=== CLEANED FEATURE LIST ===")
print(feature_cols[:8], "... (plus one-hot encoded specialties)")
print(f"Total features used: {len(feature_cols)}\n")

print("=== HONEST TEST SET METRICS ===")
print(f"AUC:       {roc_auc_score(y_test, y_prob):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

C:\Users\MSI\anaconda3\envs\zs_project\lib\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")


=== CLEANED FEATURE LIST ===
['total_claims', 'total_patients', 'total_drug_cost', 'unique_drugs', 'emails_sent', 'emails_opened', 'rep_visits', 'rep_visit_accepted'] ... (plus one-hot encoded specialties)
Total features used: 171

=== HONEST TEST SET METRICS ===
AUC:       0.9732
Precision: 0.7070
Recall:    0.9125
F1 Score:  0.7967

Confusion Matrix:
[[94809  8289]
 [ 1919 20001]]


In [2]:
import pandas as pd
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances.head(15))

unique_drugs                                                                0.267559
specialty_Endocrinology                                                     0.095091
specialty_Psychiatry                                                        0.058203
specialty_Family Practice                                                   0.052437
specialty_Nurse Practitioner                                                0.038961
specialty_Student in an Organized Health Care Education/Training Program    0.038663
specialty_Internal Medicine                                                 0.036708
specialty_Dentist                                                           0.032060
specialty_Neurology                                                         0.029696
specialty_Physician Assistant                                               0.026673
specialty_Pharmacist                                                        0.026536
specialty_Psychiatry & Neurology                                 

In [3]:
from sklearn.linear_model import LogisticRegression
specialty_cols = [c for c in feature_cols if c.startswith('specialty_')]
X_baseline = X[specialty_cols]
Xb_train, Xb_test, yb_train, yb_test = train_test_split(X_baseline, y, test_size=0.2, random_state=42, stratify=y)
lr = LogisticRegression(max_iter=1000).fit(Xb_train, yb_train)
print("Specialty-only AUC:", roc_auc_score(yb_test, lr.predict_proba(Xb_test)[:,1]))

Specialty-only AUC: 0.8616095122644276


In [4]:
import joblib

# Save the cleaned model
joblib.dump(model, 'propensity_model_v2.pkl')

# Rescore the full population with cleaned model
df_out = df_model_base.copy()
df_out['propensity_score'] = model.predict_proba(X[feature_cols].fillna(0))[:, 1]
df_out.to_csv('hcp_scored_v2.csv', index=False)
print(df_out[['Prscrbr_NPI', 'propensity_score']].describe())

        Prscrbr_NPI  propensity_score
count  6.250890e+05     625089.000000
mean   1.274708e+09          0.257110
std    1.597184e+08          0.355148
min    1.003000e+09          0.000123
25%    1.134629e+09          0.009007
50%    1.275519e+09          0.051128
75%    1.417076e+09          0.405802
max    1.548804e+09          0.998395


In [5]:
import joblib

# Save the cleaned model
joblib.dump(model, 'propensity_model_v2.pkl')

# Rescore the full population with cleaned model
df_out = df_model_base.copy()
df_out['propensity_score'] = model.predict_proba(X[feature_cols].fillna(0))[:, 1]
df_out.to_csv('hcp_scored_v2.csv', index=False)
print(df_out[['Prscrbr_NPI', 'propensity_score']].describe())

        Prscrbr_NPI  propensity_score
count  6.250890e+05     625089.000000
mean   1.274708e+09          0.257110
std    1.597184e+08          0.355148
min    1.003000e+09          0.000123
25%    1.134629e+09          0.009007
50%    1.275519e+09          0.051128
75%    1.417076e+09          0.405802
max    1.548804e+09          0.998395


In [6]:
non_prescribers = df_out[df_out['prescribes_glp1'] == 0].copy()
total_calls = 61 * 50

# Baseline: what a rep would pick using specialty alone (top specialties by known conversion rate)
# Model: top by propensity_score
model_driven = non_prescribers.nlargest(total_calls, 'propensity_score')

# For baseline, use the specialty-only model's predicted probs (already trained above)
non_prescribers_baseline_idx = non_prescribers.index
X_baseline_full = X_baseline.loc[non_prescribers_baseline_idx]
non_prescribers['specialty_only_score'] = lr.predict_proba(X_baseline_full)[:, 1]
baseline_driven = non_prescribers.nlargest(total_calls, 'specialty_only_score')

print("Model avg propensity:", model_driven['propensity_score'].mean())
print("Specialty-baseline avg propensity:", baseline_driven['propensity_score'].mean())
print("Real lift over specialty targeting:", 
      (model_driven['propensity_score'].mean() / baseline_driven['propensity_score'].mean() - 1))

Model avg propensity: 0.9773561
Specialty-baseline avg propensity: 0.46060964
Real lift over specialty targeting: 1.121875


In [7]:
total_calls = 61 * 50
cost_per_visit = 150

b_conv = round(0.46060964 * total_calls)
m_conv = round(0.9773561 * total_calls)
b_cost = (total_calls * cost_per_visit) / b_conv
m_cost = (total_calls * cost_per_visit) / m_conv

print(f"Specialty-baseline converters: {b_conv}")
print(f"Model converters: {m_conv}")
print(f"Baseline cost/conversion: ${b_cost:,.0f}")
print(f"Model cost/conversion: ${m_cost:,.0f}")
print(f"Savings per conversion: ${b_cost - m_cost:,.0f}")

Specialty-baseline converters: 1405
Model converters: 2981
Baseline cost/conversion: $326
Model cost/conversion: $153
Savings per conversion: $172


In [9]:
print(pd.read_csv('hcp_scored_v2.csv').columns.tolist())
print(pd.read_csv('crm_layer.csv').columns.tolist())

['Prscrbr_NPI', 'total_claims', 'total_patients', 'total_drug_cost', 'unique_drugs', 'specialty', 'city', 'state', 'glp1_claims', 'glp1_patients', 'prescribes_glp1', 'emails_sent', 'emails_opened', 'rep_visits', 'rep_visit_accepted', 'webinars_invited', 'webinars_attended', 'digital_score', 'propensity_score']
['Prscrbr_NPI', 'emails_sent', 'emails_opened', 'rep_visits', 'rep_visit_accepted', 'webinars_invited', 'webinars_attended', 'digital_score']


In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df = pd.read_csv('hcp_scored_v2.csv')

features = ['total_claims','total_patients','total_drug_cost',
            'unique_drugs','propensity_score','digital_score',
            'rep_visit_accepted','webinars_attended']

X = df[features].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

km = KMeans(n_clusters=5, random_state=42, n_init=10)
df['segment'] = km.fit_predict(X_scaled)

profile = df.groupby('segment')[features].mean().round(2)
print(profile)
print("\nCount per segment:")
print(df['segment'].value_counts())

         total_claims  total_patients  total_drug_cost  unique_drugs  \
segment                                                                
0              426.10          150.79        103799.25          9.46   
1             3105.14          782.63        262770.00         65.88   
2            11936.89         2890.90       1415503.82        136.17   
3              410.63          147.16        100050.16          9.12   
4           209328.78       186367.22      41814056.71         52.11   

         propensity_score  digital_score  rep_visit_accepted  \
segment                                                        
0                    0.10          45.59                1.60   
1                    0.89          30.28                1.61   
2                    0.89          30.74                1.61   
3                    0.10          19.42                1.59   
4                    0.46          28.27                2.33   

         webinars_attended  
segment          

In [11]:
# Programmatic segment labeling from profile
profile = df.groupby('segment')[features].mean()

# Exclude outlier segment (fewer than 50 HCPs)
counts = df['segment'].value_counts()
valid_segments = counts[counts >= 50].index
profile_valid = profile.loc[valid_segments]

# Rank-based labeling
champions_seg    = profile_valid['total_claims'].idxmax()
glp1_rising_seg  = profile_valid.drop(champions_seg)['propensity_score'].idxmax()
digital_first_seg = profile_valid.drop([champions_seg, glp1_rising_seg])['digital_score'].idxmax()
low_priority_seg = profile_valid.drop([champions_seg, glp1_rising_seg, digital_first_seg]).index[0]
outlier_seg      = counts[counts < 50].index[0]

label_map = {
    champions_seg:     'Champions',
    glp1_rising_seg:   'GLP1_Rising',
    digital_first_seg: 'Digital_First',
    low_priority_seg:  'Low_Priority',
    outlier_seg:       'Institutional'
}

df['segment_name'] = df['segment'].map(label_map)

print("Label mapping:", label_map)
print("\nSegment distribution:")
print(df['segment_name'].value_counts())

df.to_csv('hcp_segmented_v2.csv', index=False)
print("Saved: hcp_segmented_v2.csv")


Label mapping: {np.int32(2): 'Champions', np.int32(1): 'GLP1_Rising', np.int32(0): 'Digital_First', np.int32(3): 'Low_Priority', np.int32(4): 'Institutional'}

Segment distribution:
segment_name
Low_Priority     282427
Digital_First    218238
GLP1_Rising      101000
Champions         23415
Institutional         9
Name: count, dtype: int64
Saved: hcp_segmented_v2.csv


In [12]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
import networkx as nx

df = pd.read_csv('hcp_segmented_v2.csv')

# Build network only on high-signal HCPs (exclude Low_Priority + Institutional)
df_net = df[df['segment_name'].isin(['Champions','GLP1_Rising','Digital_First'])]\
           .nlargest(5000, 'propensity_score').reset_index(drop=True)

print(f"Network population: {df_net.shape[0]}")
print(df_net['segment_name'].value_counts())

# KNN graph on GLP-1 relevant features only
sim_features = ['propensity_score','unique_drugs','total_claims','digital_score']
X_sim = df_net[sim_features].fillna(0).values
X_sim = (X_sim - X_sim.mean(axis=0)) / (X_sim.std(axis=0) + 1e-9)

nn = NearestNeighbors(n_neighbors=6)
nn.fit(X_sim)
distances, indices = nn.kneighbors(X_sim)

G = nx.Graph()
for i in range(len(df_net)):
    G.add_node(i, npi=df_net.loc[i,'Prscrbr_NPI'], specialty=df_net.loc[i,'specialty'])

for i in range(len(df_net)):
    for j_idx in range(1, 6):
        j = indices[i][j_idx]
        if df_net.loc[i,'specialty'] == df_net.loc[j,'specialty']:
            G.add_edge(i, j)

print(f"\nNodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}")

# PageRank
pagerank_scores = nx.pagerank(G, alpha=0.85)
df_net['pagerank'] = df_net.index.map(pagerank_scores)

# Merge back to full df
df_full = df.merge(df_net[['Prscrbr_NPI','pagerank']], on='Prscrbr_NPI', how='left')
df_full['pagerank'] = df_full['pagerank'].fillna(0)

threshold = df_net['pagerank'].quantile(0.95)
df_full['is_kol'] = (df_full['pagerank'] >= threshold).astype(int)

print(f"\nKOLs identified: {df_full['is_kol'].sum()}")
print("KOL specialty breakdown:")
print(df_full[df_full['is_kol']==1]['specialty'].value_counts().head(10))

df_full.to_csv('hcp_kol_v2.csv', index=False)
print("\nSaved: hcp_kol_v2.csv")

Network population: 5000
segment_name
Champions        3375
GLP1_Rising      1618
Digital_First       7
Name: count, dtype: int64

Nodes: 5000 | Edges: 16061

KOLs identified: 250
KOL specialty breakdown:
specialty
Family Practice    155
Endocrinology       95
Name: count, dtype: int64

Saved: hcp_kol_v2.csv


In [13]:
import pandas as pd
import numpy as np

df = pd.read_csv('hcp_kol_v2.csv')
np.random.seed(42)

channels = ['email', 'rep_visit', 'webinar']

# Response probabilities grounded in segment behavior
true_probs = {
    'Champions':     {'email': 0.25, 'rep_visit': 0.55, 'webinar': 0.30},
    'GLP1_Rising':   {'email': 0.30, 'rep_visit': 0.50, 'webinar': 0.35},
    'Digital_First': {'email': 0.45, 'rep_visit': 0.20, 'webinar': 0.40},
    'Low_Priority':  {'email': 0.10, 'rep_visit': 0.08, 'webinar': 0.12},
    'Institutional': {'email': 0.15, 'rep_visit': 0.25, 'webinar': 0.20}
}

n_trials = 5
records = []
for idx, row in df.iterrows():
    seg = row['segment_name']
    npi = row['Prscrbr_NPI']
    probs = true_probs.get(seg, true_probs['Low_Priority'])
    for ch in channels:
        successes = np.random.binomial(n_trials, probs[ch])
        records.append({
            'Prscrbr_NPI': npi,
            'segment_name': seg,
            'channel': ch,
            'trials': n_trials,
            'successes': successes
        })

interaction_log = pd.DataFrame(records)

# Aggregate and pivot
agg = interaction_log.groupby(['Prscrbr_NPI','channel']).agg(
    trials=('trials','sum'),
    successes=('successes','sum')
).reset_index()
agg['failures'] = agg['trials'] - agg['successes']

pivot_succ = agg.pivot(index='Prscrbr_NPI', columns='channel', values='successes').fillna(0)
pivot_fail = agg.pivot(index='Prscrbr_NPI', columns='channel', values='failures').fillna(0)

# Thompson Sampling — pick best channel per HCP
best_actions = []
for npi in pivot_succ.index:
    sampled = {ch: np.random.beta(pivot_succ.loc[npi,ch]+1, pivot_fail.loc[npi,ch]+1) 
               for ch in channels}
    best_actions.append({'Prscrbr_NPI': npi, 'next_best_action': max(sampled, key=sampled.get)})

nba_df = pd.DataFrame(best_actions)
df = df.merge(nba_df, on='Prscrbr_NPI', how='left')

print("NBA Distribution:")
print(df['next_best_action'].value_counts())
print(f"\nShape: {df.shape}")

df.to_csv('hcp_nba_v2.csv', index=False)
print("Saved: hcp_nba_v2.csv")

NBA Distribution:
next_best_action
webinar      221721
email        221439
rep_visit    181929
Name: count, dtype: int64

Shape: (625089, 24)
Saved: hcp_nba_v2.csv


In [14]:
import pandas as pd
import numpy as np
import pulp

df = pd.read_csv('hcp_nba_v2.csv')

# Priority score
segment_weight = {
    'Champions': 1.0, 'GLP1_Rising': 0.9, 'Digital_First': 0.6,
    'Low_Priority': 0.2, 'Institutional': 0.3
}
df['segment_weight'] = df['segment_name'].map(segment_weight).fillna(0.1)
df['priority_score'] = (
    df['propensity_score'] * 0.5 +
    df['segment_weight'] * 0.3 +
    df['is_kol'] * 0.2
)

# Channel costs — this is what justifies LP over simple sort
CHANNEL_COST = {'rep_visit': 150, 'email': 5, 'webinar': 30}
REP_WEEKLY_CAPACITY = 50
REP_WEEKLY_BUDGET   = 3000   # $3000/week per rep — realistic pharma field budget
MIN_KOL_CALLS       = 3      # at least 3 KOLs per territory per week
MAX_REP_VISITS      = 20     # rep visits are time-heavy; cap at 20/week

territories = df['state'].dropna().unique()
all_selected = []

for state in territories:
    sdf = df[df['state'] == state].reset_index(drop=True)
    n = len(sdf)
    capacity = min(REP_WEEKLY_CAPACITY, n)

    prob = pulp.LpProblem(f"CallPlan_{state}", pulp.LpMaximize)
    x = [pulp.LpVariable(f"x_{i}", cat='Binary') for i in range(n)]

    # Objective: maximize total priority score
    prob += pulp.lpSum(x[i] * sdf.loc[i,'priority_score'] for i in range(n))

    # Constraint 1: call capacity
    prob += pulp.lpSum(x) <= capacity

    # Constraint 2: weekly budget cap (this is what makes LP necessary)
    prob += pulp.lpSum(
        x[i] * CHANNEL_COST.get(sdf.loc[i,'next_best_action'], 30)
        for i in range(n)
    ) <= REP_WEEKLY_BUDGET

    # Constraint 3: max rep visits (time constraint)
    rep_visit_idx = sdf[sdf['next_best_action'] == 'rep_visit'].index.tolist()
    if rep_visit_idx:
        prob += pulp.lpSum(x[i] for i in rep_visit_idx) <= MAX_REP_VISITS

    # Constraint 4: minimum KOL calls
    kol_idx = sdf[sdf['is_kol'] == 1].index.tolist()
    if len(kol_idx) >= MIN_KOL_CALLS:
        prob += pulp.lpSum(x[i] for i in kol_idx) >= MIN_KOL_CALLS

    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    sdf['selected'] = [int(x[i].varValue or 0) for i in range(n)]
    all_selected.append(sdf)

call_plan = pd.concat(all_selected, ignore_index=True)

selected = call_plan[call_plan['selected'] == 1]
print(f"Total HCPs selected: {selected.shape[0]}")
print(f"Total weekly cost: ${(selected['next_best_action'].map(CHANNEL_COST).sum()):,.0f}")
print(f"KOLs in plan: {selected['is_kol'].sum()}")
print(f"Channel breakdown:\n{selected['next_best_action'].value_counts()}")
print(f"Segment breakdown:\n{selected['segment_name'].value_counts()}")

call_plan.to_csv('hcp_final_v2.csv', index=False)
print("\nSaved: hcp_final_v2.csv")

Total HCPs selected: 2867
Total weekly cost: $169,880
KOLs in plan: 250
Channel breakdown:
next_best_action
email        994
webinar      967
rep_visit    906
Name: count, dtype: int64
Segment breakdown:
segment_name
Champions        2221
GLP1_Rising       433
Low_Priority      108
Digital_First     105
Name: count, dtype: int64

Saved: hcp_final_v2.csv


In [16]:
import os
print(os.path.getsize('hcp_final_v2.csv') / 1e6, "MB")

96.388337 MB


In [17]:
import pandas as pd

df = pd.read_csv('hcp_final_v2.csv')
print("Current columns:", df.columns.tolist())
print("Current shape:", df.shape)

# Keep only what app.py actually uses
keep_cols = [
    'Prscrbr_NPI', 'specialty', 'state', 'city',
    'segment_name', 'propensity_score', 'is_kol',
    'next_best_action', 'priority_score', 'prescribes_glp1'
]

df_app = df[keep_cols]
df_app.to_csv('hcp_app.csv', index=False)

import os
print(f"Trimmed size: {os.path.getsize('hcp_app.csv') / 1e6:.2f} MB")
print(f"Shape: {df_app.shape}")

Current columns: ['Prscrbr_NPI', 'total_claims', 'total_patients', 'total_drug_cost', 'unique_drugs', 'specialty', 'city', 'state', 'glp1_claims', 'glp1_patients', 'prescribes_glp1', 'emails_sent', 'emails_opened', 'rep_visits', 'rep_visit_accepted', 'webinars_invited', 'webinars_attended', 'digital_score', 'propensity_score', 'segment', 'segment_name', 'pagerank', 'is_kol', 'next_best_action', 'segment_weight', 'priority_score', 'selected']
Current shape: (625089, 27)
Trimmed size: 57.91 MB
Shape: (625089, 10)


In [19]:
import os
print(os.path.getsize('hcp_app.csv') / 1e6, "MB")

57.913942 MB
